# Execução do RNAmining nas espécies

Este notebook existe para rodar o RNAmining nas espécies do dataset preparado e gerar as métricas de avaliação.

Aqui não há preparação de dados. Assume-se que os FASTAs já estão organizados, padronizados e prontos para uso.

## O que ele faz

O notebook executa as seguintes etapas:

1. **Definição de paths com `pathlib`**  
   Centraliza os caminhos para os FASTAs de entrada e para as saídas do RNAmining.

2. **Mapeamento das espécies**  
   Identifica quais espécies serão processadas e associa cada uma aos seus arquivos de entrada.

3. **Execução do RNAmining**  
   Para cada espécie:
   - roda o RNAmining via CLI  
   - define corretamente o `organism_name`  
   - salva a saída no diretório correspondente  

4. **Coleta das predições**  
   Lê os arquivos `predictions.txt` gerados para cada espécie.

5. **Extração do ground truth**  
   Obtém os rótulos reais diretamente dos FASTAs (via header).

6. **Alinhamento entre predição e ground truth**  
   Garante correspondência correta entre IDs de sequência.

7. **Cálculo de métricas**  
   Calcula, por espécie:
   - accuracy  
   - precision  
   - recall  
   - f1  
   - mcc  

8. **Geração dos resultados finais**  
   Consolida os resultados em tabela para análise e comparação.

## Resultado esperado

Ao final, o notebook gera:
- predições do RNAmining por espécie  
- métricas individuais por espécie  
- base pronta para comparação com outras ferramentas

In [1]:
from pathlib import Path
import subprocess
import math
import csv
import os
import pandas as pd

## 1. Paths principais

In [2]:
# Pasta base do projeto RNAmining
BASE_RNAMINING = Path(os.environ.get("RNAMINING_DIR"))

# Script oficial do RNAmining
SCRIPT_RNAMINING = BASE_RNAMINING / "volumes" / "rnamining-front" / "assets" / "scripts" / "rnamining.py"

# Pasta com os FASTAs de teste do S5
DADOS_RNAMINING = BASE_RNAMINING / "volumes" / "rnamining-front" / "data" / "S5_File"  / "Model_Organisms"

# Pasta onde os resultados serão salvos
OUT_DIR = BASE_RNAMINING / "rnamining_out"

# Tipo de predição usado pelo RNAmining
PREDICTION_TYPE = "coding_prediction"

## 2. Funções auxiliares

In [3]:
# Converte o nome do arquivo para o nome do organismo esperado pelo RNAMining
# Basicamente, só retira o final do nome, para preservar o nome da espécie
def extrair_nome_organismo(caminho_fasta):
    nome = caminho_fasta.stem.replace("_test", "")
    return nome


# Lê o FASTA de teste e monta o ground truth
# Regra usada no S4:
# - cds -> 1
# - ncrna -> 0
def ler_gt_fasta(caminho_fasta):
    gt = {}

    with open(caminho_fasta, "r") as f:
        for linha in f:
            # As linhas que começam com > marcam o header das sequências
            if linha.startswith(">"):
                header = linha[1:].strip()
                seq_id = header.split()[0]
                texto = header.lower()

                if " cds " in f" {texto} ":
                    gt[seq_id] = 1
                elif " ncrna " in f" {texto} ":
                    gt[seq_id] = 0

    return gt


# Normaliza a label prevista
def normalizar_label_predicao(label):
    texto = str(label).strip().lower()

    if texto in {"coding", "coding_rna", "cod", "mrna"}:
        return 1
    elif texto in {"non-coding", "noncoding", "noncodingrna", "lncrna", "ncrna"}:
        return 0
    else:
        return None


# Lê o predictions.txt do RNAMining
# Para casar com o GT, é utilizado apenas o primeiro token do header
def ler_predicoes_rnamining(caminho_pred):
    pred = {}

    with open(caminho_pred, "r") as f:
        linhas = [linha.rstrip("\n") for linha in f]

    for linha in linhas:
        linha = linha.strip()

        if not linha:
            continue
        if linha.startswith("RNAMining Predictions"):
            continue
        if linha.startswith("Prediction Type:"):
            continue
        if linha.startswith("Name of the Organism:"):
            continue
        if linha.startswith("Sequence ID"):
            continue

        partes = linha.split("\t")

        if len(partes) < 2:
            continue

        header_original = partes[0].strip()
        seq_id = header_original.split()[0]

        label_prevista = partes[1].strip()
        label_prevista = normalizar_label_predicao(label_prevista)

        if label_prevista is not None:
            pred[seq_id] = label_prevista

    return pred


# Calcula TP, TN, FP, FN e métricas
def calcular_metricas(gt, pred):
    ids_comuns = sorted(set(gt) & set(pred))

    tp = tn = fp = fn = 0

    for seq_id in ids_comuns:
        verdadeiro = gt[seq_id]
        previsto = pred[seq_id]

        if previsto == 1 and verdadeiro == 1:
            tp += 1
        elif previsto == 0 and verdadeiro == 0:
            tn += 1
        elif previsto == 1 and verdadeiro == 0:
            fp += 1
        elif previsto == 0 and verdadeiro == 1:
            fn += 1

    total = tp + tn + fp + fn

    accuracy = (tp + tn) / total if total != 0 else 0
    precision = tp / (tp + fp) if (tp + fp) != 0 else 0
    recall = tp / (tp + fn) if (tp + fn) != 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) != 0 else 0

    denominador = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = ((tp * tn) - (fp * fn)) / denominador if denominador != 0 else 0

    return {
        "n_total_eval": len(gt),
        "n_pred_rows": len(pred),
        "n_merged": len(ids_comuns),
        "n_missing_pred": len(set(gt) - set(pred)),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mcc": mcc,
    }

## 3. Conferência rápida dos arquivos de teste

In [4]:
arquivos_teste = sorted(DADOS_RNAMINING.glob("*_test.fa"))
len(arquivos_teste), [arquivo.name for arquivo in arquivos_teste[:5]]

(16,
 ['Anolis_carolinensis_test.fa',
  'Chrysemys_picta_bellii_test.fa',
  'Crocodylus_porosus_test.fa',
  'Danio_rerio_test.fa',
  'Eptatretus_burgeri_test.fa'])

## 4. Rodar o RNAMining

In [5]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_EXEC = OUT_DIR / "execucao_rnamining.csv"

In [6]:
with open(CSV_EXEC, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "species",
        "organism_name",
        "status",
        "test_file",
        "output_dir",
        "prediction_file",
        "returncode"
    ])

for caminho_teste in arquivos_teste:
    nome_curto = caminho_teste.stem.replace("_test", "").split("_")[0].lower()
    organism_name = extrair_nome_organismo(caminho_teste)

    pasta_saida = OUT_DIR / nome_curto
    pasta_saida.mkdir(parents=True, exist_ok=True)

    arquivo_pred = pasta_saida / "predictions.txt"

    comando = [
        "python",
        str(SCRIPT_RNAMINING),
        "-f", str(caminho_teste),
        "-organism_name", organism_name,
        "-prediction_type", PREDICTION_TYPE,
        "-output_folder", str(pasta_saida)
    ]

    print(f"Rodando {organism_name}")
    resultado = subprocess.run(comando, capture_output=True, text=True)

    status_exec = "ok" if resultado.returncode == 0 else "erro"

    with open(CSV_EXEC, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            nome_curto,
            organism_name,
            status_exec,
            str(caminho_teste),
            str(pasta_saida),
            str(arquivo_pred),
            resultado.returncode
        ])

Rodando Anolis_carolinensis
Rodando Chrysemys_picta_bellii
Rodando Crocodylus_porosus
Rodando Danio_rerio
Rodando Eptatretus_burgeri
Rodando Gallus_gallus
Rodando Homo_sapiens
Rodando Latimeria_chalumnae
Rodando Monodelphis_domestica
Rodando Mus_musculus
Rodando Notechis_scutatus
Rodando Ornithorhynchus_anatinus
Rodando Petromyzon_marinus
Rodando Rattus_norvegicus
Rodando Sphenodon_punctatus
Rodando Xenopus_tropicalis


## 5. Ver resumo das execuções

In [7]:
pd.read_csv(CSV_EXEC)

,species,organism_name,status,test_file,output_dir,prediction_file,returncode
0,anolis,Anolis_carolinensis,ok,/home/samuel/projects/ARIA/RNAmining-updated/v...,/home/samuel/projects/ARIA/RNAmining-updated/r...,/home/samuel/projects/ARIA/RNAmining-updated/r...,0
1,chrysemys,Chrysemys_picta_bellii,ok,/home/samuel/projects/ARIA/RNAmining-updated/v...,/home/samuel/projects/ARIA/RNAmining-updated/r...,/home/samuel/projects/ARIA/RNAmining-updated/r...,0
2,crocodylus,Crocodylus_porosus,ok,/home/samuel/projects/ARIA/RNAmining-updated/v...,/home/samuel/projects/ARIA/RNAmining-updated/r...,/home/samuel/projects/ARIA/RNAmining-updated/r...,0
3,danio,Danio_rerio,ok,/home/samuel/projects/ARIA/RNAmining-updated/v...,/home/samuel/projects/ARIA/RNAmining-updated/r...,/home/samuel/projects/ARIA/RNAmining-updated/r...,0
4,eptatretus,Eptatretus_burgeri,ok,/home/samuel/projects/ARIA/RNAmining-updated/v...,/home/samuel/projects/ARIA/RNAmining-updated/r...,/home/samuel/projects/ARIA/RNAmining-updated/r...,0
5,gallus,Gallus_gallus,ok,/home/samuel/projects/ARIA/RNAmining-updated/v...,/home/samuel/projects/ARIA/RNAmining-updated/r...,/home/samuel/projects/ARIA/RNAmining-updated/r...,0
6,homo,Homo_sapiens,ok,/home/samuel/projects/ARIA/RNAmining-updated/v...,/home/samuel/projects/ARIA/RNAmining-updated/r...,/home/samuel/projects/ARIA/RNAmining-updated/r...,0
7,latimeria,Latimeria_chalumnae,ok,/home/samuel/projects/ARIA/RNAmining-updated/v...,/home/samuel/projects/ARIA/RNAmining-updated/r...,/home/samuel/projects/ARIA/RNAmining-updated/r...,0
8,monodelphis,Monodelphis_domestica,ok,/home/samuel/projects/ARIA/RNAmining-updated/v...,/home/samuel/projects/ARIA/RNAmining-updated/r...,/home/samuel/projects/ARIA/RNAmining-updated/r...,0
9,mus,Mus_musculus,ok,/home/samuel/projects/ARIA/RNAmining-updated/v...,/home/samuel/projects/ARIA/RNAmining-updated/r...,/home/samuel/projects/ARIA/RNAmining-updated/r...,0


## 6. Calcular métricas por espécie

In [8]:
CSV_METRICS = OUT_DIR / "metrics_rnamining.csv"

with open(CSV_METRICS, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "species",
        "organism_name",
        "status",
        "n_total_eval",
        "n_pred_rows",
        "n_merged",
        "n_missing_pred",
        "tp",
        "tn",
        "fp",
        "fn",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "mcc"
    ])

for caminho_teste in arquivos_teste:
    nome_curto = caminho_teste.stem.replace("_test", "").split("_")[0].lower()
    organism_name = extrair_nome_organismo(caminho_teste)

    caminho_pred = OUT_DIR / nome_curto / "predictions.txt"

    gt = ler_gt_fasta(caminho_teste)
    pred = ler_predicoes_rnamining(caminho_pred)

    metricas = calcular_metricas(gt, pred)

    with open(CSV_METRICS, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            nome_curto,
            organism_name,
            "ok",
            metricas["n_total_eval"],
            metricas["n_pred_rows"],
            metricas["n_merged"],
            metricas["n_missing_pred"],
            metricas["tp"],
            metricas["tn"],
            metricas["fp"],
            metricas["fn"],
            metricas["accuracy"],
            metricas["precision"],
            metricas["recall"],
            metricas["f1"],
            metricas["mcc"]
        ])

## 7. Ver tabela final de métricas

In [9]:
pd.read_csv(CSV_METRICS)

,species,organism_name,status,n_total_eval,n_pred_rows,n_merged,n_missing_pred,tp,tn,fp,fn,accuracy,precision,recall,f1,mcc
0,anolis,Anolis_carolinensis,ok,1904,1904,1904,0,931,898,54,21,0.960609,0.945178,0.977941,0.961280,0.921772
1,chrysemys,Chrysemys_picta_bellii,ok,2816,2816,2816,0,1381,1371,37,27,0.977273,0.973907,0.980824,0.977353,0.954570
2,crocodylus,Crocodylus_porosus,ok,1848,1848,1848,0,907,902,22,17,0.978896,0.976319,0.981602,0.978953,0.957806
3,danio,Danio_rerio,ok,3246,3246,3246,0,1603,1571,52,20,0.977819,0.968580,0.987677,0.978035,0.955824
4,eptatretus,Eptatretus_burgeri,ok,436,436,436,0,216,210,8,2,0.977064,0.964286,0.990826,0.977376,0.954490
5,gallus,Gallus_gallus,ok,11102,11102,11102,0,5521,5506,45,30,0.993244,0.991915,0.994596,0.993254,0.986493
6,homo,Homo_sapiens,ok,81500,81500,81500,0,40562,40324,426,188,0.992466,0.989607,0.995387,0.992488,0.984949
7,latimeria,Latimeria_chalumnae,ok,1168,1168,1168,0,578,583,1,6,0.994007,0.998273,0.989726,0.993981,0.988050
8,monodelphis,Monodelphis_domestica,ok,8584,8584,8584,0,4262,4224,68,30,0.988583,0.984296,0.993010,0.988634,0.977205
9,mus,Mus_musculus,ok,26668,26668,26668,0,13268,13143,191,66,0.990363,0.985809,0.995050,0.990408,0.980769


## 8. Inspeção rápida de uma espécie

In [10]:
especie_exemplo = "anolis"

caminho_pred = OUT_DIR / especie_exemplo / "predictions.txt"
caminho_teste = DADOS_RNAMINING / "Anolis_carolinensis_test.fa"

print("Arquivo de predição:")
print(caminho_pred)

print("\nPrimeiras linhas do predictions.txt:")
with open(caminho_pred, "r") as f:
    for i, linha in enumerate(f):
        if i == 8:
            break
        print(linha.rstrip())

print("\nResumo do GT e da predição:")
gt_exemplo = ler_gt_fasta(caminho_teste)
pred_exemplo = ler_predicoes_rnamining(caminho_pred)

print("GT:", len(gt_exemplo))
print("Pred:", len(pred_exemplo))
print("IDs em comum:", len(set(gt_exemplo) & set(pred_exemplo)))

Arquivo de predição:
/home/samuel/projects/ARIA/RNAmining-updated/rnamining_out/anolis/predictions.txt

Primeiras linhas do predictions.txt:
RNAMining Predictions
Prediction Type: coding_prediction
Name of the Organism: Anolis_carolinensis
Sequence ID 	 Predictions:

ENSACAT00000047339.1 ncrna primary_assembly:AnoCar2.0v2:GL343279.1:571173:573905:-1 gene:ENSACAG00000044519.1 gene_biotype:lncRNA transcript_biotype:lncRNA class:noncoding	coding	0.9347108
ENSACAT00000040374.1 ncrna primary_assembly:AnoCar2.0v2:2:155235407:155235544:1 gene:ENSACAG00000043409.1 gene_biotype:snRNA transcript_biotype:snRNA gene_symbol:U2 description:U2 spliceosomal RNA [Source:RFAM;Acc:RF00004] class:noncoding	non-coding	0.99994904
ENSACAT00000037927.1 cds primary_assembly:AnoCar2.0v2:GL343210.1:2128849:2131846:-1 gene:ENSACAG00000037137.1 gene_biotype:protein_coding transcript_biotype:protein_coding gene_symbol:SNTN description:sentan, cilia apical structure protein [Source:UniProtKB Gene Name;Acc:A0A803TSE4